In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, struct, to_json, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

In [32]:
spark = SparkSession.builder \
    .appName("TrasactionNotifier") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .getOrCreate()

In [33]:
user_df = spark.read.csv("data/user_table.csv", header= True, inferSchema= True)

In [34]:
tx_schema = StructType({
    StructField("tx_id", IntegerType(), True),
     StructField("userId", IntegerType(), True),
     StructField("amount", DoubleType(), True),
     StructField("timestamp", DoubleType(), True)
})

kafka_stream = spark.readStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("subscribe", "trasaction-notification") \
 .load()

In [35]:
parsed_stream = kafka_stream.select(
    from_json(
        col("value").cast("string"), 
        tx_schema
    ).alias("tx")
).select("tx.*")

In [36]:
transact = parsed_stream.filter(col("amount") > 5000.0)
fraud_trx = parsed_stream.filter(col("amount") > 10000.0)

In [37]:
t5k_fraud = transact.join(user_df, "userId")
t10k_fraud = fraud_trx.join(user_df, "userId")

In [38]:
output_stream_5k = t5k_fraud \
 .withColumn("value", to_json(struct("*")).cast("string")) \
 .select("value")

output_stream_10k = t10k_fraud \
 .withColumn("value", to_json(struct("*")).cast("string")) \
 .select("value")

In [ ]:
query = output_stream_10k.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notify") \
 .option("checkpointLocation", "/tmp/checkpoints0") \
 .start()

audit_query = output_stream_5k.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic","audit-log") \
    .option("checkpointLocation", "/tmp/checkpoints1") \
    .start()

query.awaitTermination()
audit_query.awaitTermination()

26/06/19 06:09:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:09:10 WARN StreamingQueryManager: Stopping existing streaming query [id=499b5521-7bc4-4e8a-ac9d-65b907149bbd, runId=ec538e9f-2c94-42d9-ab65-9db0999c5cce], as a new run is being started.
26/06/19 06:09:11 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/19 06:09:11 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:09:11 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
